In [ ]:
import numpy as np
from scipy.special import expit
from scipy.optimize import minimize
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
def sample_truncnorm(mean, sd, low, high, size, rng):
    """Draw `size` iid N(mean, sd^2) samples truncated to [low, high] via rejection sampling."""
    out = np.empty(size, dtype=float)
    filled = 0
    while filled < size:
        m = int(1.5 * (size - filled) + 32)
        x = rng.normal(loc=mean, scale=sd, size=m)
        x = x[(x >= low) & (x <= high)]
        take = min(x.size, size - filled)
        if take:
            out[filled:filled+take] = x[:take]
            filled += take
    return out

def sample_beta_sparse_truncnorm(n, M, p0=0.3, mean=-1.0, sd=0.5, rng=None):
    """Sample a sparse parameter vector: each coordinate is 0 with probability p0,
    otherwise drawn from N(mean, sd^2) truncated to [-M, M]."""
    rng = np.random.default_rng() if rng is None else rng
    beta = np.zeros(n, dtype=float)
    mask = rng.random(n) > p0
    k = int(mask.sum())
    if k:
        beta[mask] = sample_truncnorm(mean, sd, -M, M, k, rng)
    return beta


In [ ]:
def discrete_laplace_noise(eps_per_coordinate, size, rng=None):
    """Sample iid discrete Laplace noise with parameter a = exp(-eps_per_coordinate).

    For an r-uniform hypergraph degree sequence, set eps_per_coordinate = eps / r
    so that the released degrees satisfy eps-edge local differential privacy.
    """
    rng = np.random.default_rng() if rng is None else rng
    a = np.exp(-eps_per_coordinate)
    mag = rng.geometric(p=1 - a, size=size) - 1
    sign = rng.choice([-1, 1], size=size)
    return mag * sign


In [ ]:
def triu_pairs(n):
    """Return the index pairs (j, k) with j < k for n vertices."""
    j_all, k_all = np.triu_indices(n, k=1)
    return j_all.astype(np.int32), k_all.astype(np.int32)

def masks_excluding_vertex(n, j_all, k_all):
    """For each vertex i, return a boolean mask selecting pairs (j, k) not containing i."""
    masks = []
    for i in range(n):
        masks.append((j_all != i) & (k_all != i))
    return masks


In [ ]:
def sample_degrees_3uniform(beta, rng=None, chunk_size_pairs=250_000):
    """Exactly sample the degree sequence of a 3-uniform hypergraph beta-model,
    where each triple (i, j, k) is an independent hyperedge with probability
    sigmoid(beta_i + beta_j + beta_k)."""
    rng = np.random.default_rng() if rng is None else rng
    n = beta.size
    j_all, k_all = triu_pairs(n)
    pair_sum = beta[j_all] + beta[k_all]
    exclude_masks = masks_excluding_vertex(n, j_all, k_all)

    d = np.zeros(n, dtype=np.int64)

    for i in range(n):
        mask = exclude_masks[i]
        jj = j_all[mask]
        kk = k_all[mask]
        ps = pair_sum[mask]
        m = ps.size

        start = 0
        while start < m:
            end = min(start + chunk_size_pairs, m)
            logits = beta[i] + ps[start:end]
            p = expit(logits)
            u = rng.random(p.size)
            chosen = u < p
            cnt = int(chosen.sum())
            if cnt:
                d[i] += cnt
                np.add.at(d, jj[start:end][chosen], 1)
                np.add.at(d, kk[start:end][chosen], 1)
            start = end

    return d.astype(float)


In [ ]:
class HypergraphBeta3:
    """Log-partition function, gradient, and degree-based estimators for the
    3-uniform hypergraph beta-model."""

    def __init__(self, n):
        self.n = n
        self.j_all, self.k_all = triu_pairs(n)
        self.exclude_masks = masks_excluding_vertex(n, self.j_all, self.k_all)
        self.C = (n * (n - 1) * (n - 2)) / 6.0  # binom(n,3)

    def A_and_grad(self, beta, chunk_size_pairs=250_000):
        """Compute A(beta) = sum_{i<j<k} log(1+exp(beta_i+beta_j+beta_k)) and its
        gradient (expected degrees). Each triple is visited once per participating
        vertex, so both are divided by 3 to correct for the resulting overcount."""
        n = self.n
        j_all, k_all = self.j_all, self.k_all
        pair_sum = beta[j_all] + beta[k_all]

        A = 0.0
        grad = np.zeros(n, dtype=float)

        for i in range(n):
            mask = self.exclude_masks[i]
            jj = j_all[mask]
            kk = k_all[mask]
            ps = pair_sum[mask]
            m = ps.size

            start = 0
            while start < m:
                end = min(start + chunk_size_pairs, m)
                s = beta[i] + ps[start:end]
                A += np.logaddexp(0.0, s).sum()
                p = expit(s)
                grad[i] += p.sum()
                np.add.at(grad, jj[start:end], p)
                np.add.at(grad, kk[start:end], p)
                start = end

        return A / 3.0, grad / 3.0

    def ridge_box_fit_from_degrees(self, d_obs, M, lam, beta_init=None,
                                   maxiter=80, tol=1e-5, chunk_size_pairs=250_000, verbose=False):
        """Fit beta via L-BFGS-B on the box-constrained, ridge-regularized
        negative log-likelihood given observed (possibly noisy) degrees d_obs."""
        n = self.n
        if beta_init is None:
            beta_init = np.zeros(n, dtype=float)
        bounds = [(-M, M)] * n

        def fun(x):
            A, _ = self.A_and_grad(x, chunk_size_pairs=chunk_size_pairs)
            return A - float(np.dot(d_obs, x)) + 0.5 * lam * float(np.dot(x, x))

        def jac(x):
            _, gA = self.A_and_grad(x, chunk_size_pairs=chunk_size_pairs)
            return gA - d_obs + lam * x

        res = minimize(fun, beta_init, jac=jac, method="L-BFGS-B", bounds=bounds,
                       options=dict(maxiter=maxiter, ftol=tol, gtol=tol, disp=verbose))
        return res.x, res

    def grad_ell(self, beta, d_true, chunk_size_pairs=250_000):
        """Gradient of the (unregularized) negative log-likelihood ell_n(beta) =
        (A(beta) - d_true^T beta) / C at beta, given the observed degrees d_true."""
        _, gA = self.A_and_grad(beta, chunk_size_pairs=chunk_size_pairs)
        return (gA - d_true) / self.C


In [ ]:
def central_dp_gd(model: HypergraphBeta3, d_true, n, r, M, eps, delta,
                   chunk_size_pairs=250_000, seed=None, beta0=None):
    """Differentially private gradient descent estimator for beta under central
    (eps, delta)-edge differential privacy: adds Gaussian noise to the gradient
    at each step, with step size, iteration count, and noise scale calibrated
    so the final iterate is (eps, delta)-DP."""
    rng = np.random.default_rng(seed)
    if beta0 is None:
        beta = np.zeros(n, dtype=float)
    else:
        beta = beta0.astype(float).copy()

    eta = 0.25 * n * np.exp(-2.0 * r * M)
    T_init = int(np.ceil(32.0 * (r - 1) * np.exp(4.0 * r * M) * ((r - 1) * np.log(n) + 2.0 * np.log(M))))
    T = min(T_init, 1000)
    sigma2 = 4.0 * r * T * (n ** (-2.0 * r)) * (eps ** (-2.0)) * np.log(1.0 / delta)
    sigma = np.sqrt(sigma2)

    for _ in range(T):
        if (_ + 1) % 100 == 0 or (_ + 1) == T:
           print(f"\n Iteration {_ + 1}/{T}, eps={eps}", flush=True)
        g = model.grad_ell(beta, d_true, chunk_size_pairs=chunk_size_pairs)
        z = rng.normal(0.0, sigma, size=n)
        beta = beta - eta * (g + z)
        beta = np.clip(beta, -M, M)

    return beta, dict(eta=eta, T=T, sigma=sigma)


In [ ]:
def one_replicate_cache(n=200, r=3, M=2.0,
                        p0=0.3, mean=-1.0, sd=0.5,
                        c_lam=0.1,
                        chunk_size_pairs=250_000,
                        seed=None):
    """Sample beta_true and its degree sequence once, and fit the non-private
    ridge estimator, so the result can be reused across privacy budgets."""
    rng = np.random.default_rng(seed)
    beta_true = sample_beta_sparse_truncnorm(n, M, p0=p0, mean=mean, sd=sd, rng=rng)
    d_true = sample_degrees_3uniform(beta_true, rng=rng, chunk_size_pairs=chunk_size_pairs)

    lam = c_lam * (n ** ((r - 1) / 2.0))

    model = HypergraphBeta3(n)

    beta_mle, _ = model.ridge_box_fit_from_degrees(
        d_obs=d_true, M=M, lam=lam, beta_init=np.zeros(n),
        chunk_size_pairs=chunk_size_pairs
    )

    return dict(
        beta_true=beta_true,
        d_true=d_true,
        beta_mle=beta_mle,
        lam=lam,
        model=model,
        rng=rng
    )


In [ ]:
from joblib import Parallel, delayed
import numpy as np

def one_rep_results(rep_seed, n=200, r=3, M=2.0, eps_list=(0.25,0.5,0.75,1.0,1.25,1.5),
                    delta=None, p0=0.3, mean=-1.0, sd=0.5, c_lam=0.1, chunk_size_pairs=250_000):
    """Run one replicate: fit the non-private baseline once, then the local-DP
    and central-DP estimators for each epsilon in eps_list, returning
    (eps, nmse_mle, nmse_loc, nmse_cen) tuples."""
    cache = one_replicate_cache(
        n=n, r=r, M=M, p0=p0, mean=mean, sd=sd, c_lam=c_lam,
        chunk_size_pairs=chunk_size_pairs, seed=rep_seed
    )
    beta_true = cache["beta_true"]
    d_true = cache["d_true"]
    beta_mle = cache["beta_mle"]
    lam = cache["lam"]
    model = cache["model"]
    rng = cache["rng"]

    if delta is None:
        delta = n ** (-2)

    out = []
    for eps in eps_list:
        z = discrete_laplace_noise(eps_per_coordinate=eps / r, size=n, rng=rng)
        d_loc = d_true + z
        beta_loc, _ = model.ridge_box_fit_from_degrees(
            d_obs=d_loc, M=M, lam=lam, beta_init=beta_mle,
            chunk_size_pairs=chunk_size_pairs
        )

        beta_cen, _meta = central_dp_gd(
            model=model, d_true=d_true, n=n, r=r, M=M,
            eps=eps, delta=delta,
            chunk_size_pairs=chunk_size_pairs,
            seed=rng.integers(1_000_000_000),
            beta0=beta_mle
        )

        nmse_mle = float(np.mean((beta_mle - beta_true) ** 2))
        nmse_loc = float(np.mean((beta_loc - beta_true) ** 2))
        nmse_cen = float(np.mean((beta_cen - beta_true) ** 2))

        out.append((eps, nmse_mle, nmse_loc, nmse_cen))
    return out

def parallel_sweep(R=50, n_jobs=-1, seed=12345, **kwargs):
    """Run `R` independent replicates of one_rep_results in parallel across processes."""
    rng = np.random.default_rng(seed)
    seeds = rng.integers(1_000_000_000, size=R)

    results = Parallel(n_jobs=n_jobs, prefer="processes")(
        delayed(one_rep_results)(int(s), **kwargs)
        for s in tqdm(seeds, desc="Running Replicates")
    )
    return results


In [ ]:
# Example usage
eps_list = (0.25,0.5,0.75,1.0,1.25,1.5)
res = parallel_sweep(R=50, n_jobs=20, n=200, eps_list=eps_list)
